# Combined Census Block Level Roll-Up — All States

This notebook combines the 5 individual state census block rollup CSVs (CA, GA, IL, NY, TX) into a single dataset for PostgreSQL loading.

**Input files:**
- `../output/CA_block_level_availability.csv`
- `../output/GA_block_level_availability.csv`
- `../output/IL_block_level_availability.csv`
- `../output/NY_block_level_availability.csv`
- `../output/TX_block_level_availability.csv`

**Output:** `../output/all_states_block_level_availability.csv`

## 1. Import Libraries

In [1]:
import os
import pandas as pd

## 2. Load All State CSVs

In [2]:
### Define dtype to preserve leading zeros in FIPS codes
DTYPES = {
    'block_geoid': str,
    'state_fips': str,
    'county_fips': str,
    'tract_fips': str,
    'provider_id': str
}

STATE_FILES = {
    'CA': '../output/CA_block_level_availability.csv',
    'GA': '../output/GA_block_level_availability.csv',
    'IL': '../output/IL_block_level_availability.csv',
    'NY': '../output/NY_block_level_availability.csv',
    'TX': '../output/TX_block_level_availability.csv'
}

frames = []
for state, filepath in STATE_FILES.items():
    df = pd.read_csv(filepath, dtype=DTYPES)
    print(f"{state}: {df.shape[0]:,} rows")
    frames.append(df)

combined = pd.concat(frames, ignore_index=True)
print(f"\nTotal combined rows: {combined.shape[0]:,}")
print(f"Columns: {list(combined.columns)}")
combined.head()

CA: 868,477 rows


GA: 451,534 rows


IL: 646,098 rows


NY: 546,687 rows


TX: 1,105,508 rows



Total combined rows: 3,618,304
Columns: ['block_geoid', 'state_fips', 'county_fips', 'tract_fips', 'provider_id', 'holding_company', 'technology', 'technology_name', 'max_download', 'max_upload', 'serves_residential', 'serves_business', 'low_latency']


,block_geoid,state_fips,county_fips,tract_fips,provider_id,holding_company,technology,technology_name,max_download,max_upload,serves_residential,serves_business,low_latency
0,060014001001007,06,06001,06001400100,130317,Comcast Corporation,40,Cable,2000,250,True,True,1
1,060014001001009,06,06001,06001400100,130077,AT&T Inc.,10,Copper Wire (DSL),25,5,True,True,1
2,060014001001009,06,06001,06001400100,130317,Comcast Corporation,40,Cable,1200,35,True,True,1
3,060014001001010,06,06001,06001400100,130077,AT&T Inc.,10,Copper Wire (DSL),0,0,True,True,1
4,060014001001010,06,06001,06001400100,130077,AT&T Inc.,50,Fiber to the Premises (FTTP),5000,5000,True,True,1


## 3. Summary Statistics

In [3]:
### Per-state row counts
print("=== Per-State Row Counts ===")
state_counts = combined.groupby('state_fips').size().reset_index(name='rows')
state_fips_map = {'06': 'CA', '13': 'GA', '17': 'IL', '36': 'NY', '48': 'TX'}
state_counts['state'] = state_counts['state_fips'].map(state_fips_map)
print(state_counts[['state', 'state_fips', 'rows']].to_string(index=False))
print(f"\nTotal: {combined.shape[0]:,}")

=== Per-State Row Counts ===


state state_fips    rows
   CA         06  868477
   GA         13  451534
   IL         17  646098
   NY         36  546687
   TX         48 1105508

Total: 3,618,304


In [4]:
### Unique blocks, providers per state
print("=== Unique Blocks & Providers per State ===")
state_summary = combined.groupby('state_fips').agg(
    unique_blocks=('block_geoid', 'nunique'),
    unique_providers=('provider_id', 'nunique'),
    unique_companies=('holding_company', 'nunique')
).reset_index()
state_summary['state'] = state_summary['state_fips'].map(state_fips_map)
print(state_summary[['state', 'unique_blocks', 'unique_providers', 'unique_companies']].to_string(index=False))

print(f"\n--- Overall ---")
print(f"Total unique blocks:    {combined['block_geoid'].nunique():,}")
print(f"Total unique providers: {combined['provider_id'].nunique()}")
print(f"Total unique companies: {combined['holding_company'].nunique()}")

=== Unique Blocks & Providers per State ===


state  unique_blocks  unique_providers  unique_companies
   CA         369855                90                90
   GA         177295                81                81
   IL         273943                96                96
   NY         237252                69                69
   TX         452781               150               150

--- Overall ---


Total unique blocks:    1,511,126
Total unique providers: 389


Total unique companies: 389


In [5]:
### Technology distribution across all states
print("=== Technology Distribution ===")
tech_dist = combined.groupby(['technology', 'technology_name']).size().reset_index(name='rows')
tech_dist['pct'] = (tech_dist['rows'] / tech_dist['rows'].sum() * 100).round(1)
print(tech_dist.sort_values('rows', ascending=False).to_string(index=False))

print("\n--- Technology by State ---")
tech_state = combined.groupby(['state_fips', 'technology_name']).size().unstack(fill_value=0)
tech_state.index = tech_state.index.map(state_fips_map)
print(tech_state)

=== Technology Distribution ===


 technology              technology_name    rows  pct
         50 Fiber to the Premises (FTTP) 1406215 38.9
         40                        Cable 1350574 37.3
         10            Copper Wire (DSL)  861515 23.8

--- Technology by State ---


technology_name   Cable  Copper Wire (DSL)  Fiber to the Premises (FTTP)
state_fips                                                              
CA               348914             250495                        269068
GA               147413             120948                        183173
IL               256355             200677                        189066
NY               218954              36295                        291438
TX               378938             253100                        473470


In [6]:
### Top 20 providers (holding companies) across all states
print("=== Top 20 Providers (by row count) ===")
top_providers = combined['holding_company'].value_counts().head(20).reset_index()
top_providers.columns = ['holding_company', 'rows']
top_providers['pct'] = (top_providers['rows'] / combined.shape[0] * 100).round(1)
print(top_providers.to_string(index=False))

=== Top 20 Providers (by row count) ===


                        holding_company   rows  pct
                              AT&T Inc. 918774 25.4
                 Charter Communications 664447 18.4
                    Comcast Corporation 443251 12.3
    Frontier Communications Corporation 255603  7.1
                             Altice USA 232465  6.4
                       Uniti Group Inc. 118736  3.3
            Verizon Communications Inc.  98358  2.7
                   Radiate Holdings, LP  78616  2.2
              Delta Communications, LLC  54584  1.5
          Mediacom Communications Corp.  51819  1.4
                        Cable One, Inc.  42463  1.2
                 Metronet Holdings, LLC  35815  1.0
               Cox Communications, Inc.  34948  1.0
                   Fidium Holdings, LLC  31463  0.9
                 Connect Holding II LLC  29514  0.8
Foremost Telecommunications Corporation  22456  0.6
                   Conexon Connect, LLC  18177  0.5
                     Sonic Telecom, LLC  16131  0.4
            

## 4. Validation Checks

In [7]:
print("=== VALIDATION REPORT ===")
print()

# 1. Null block_geoid
null_geoid = combined['block_geoid'].isna().sum()
print(f"[{'PASS' if null_geoid == 0 else 'FAIL'}] Null block_geoid: {null_geoid}")

# 2. block_geoid length (all should be 15 digits)
bad_len = (combined['block_geoid'].str.len() != 15).sum()
print(f"[{'PASS' if bad_len == 0 else 'FAIL'}] Invalid block_geoid length: {bad_len}")

# 3. State FIPS values (should only be 06, 13, 17, 36, 48)
expected_fips = {'06', '13', '17', '36', '48'}
actual_fips = set(combined['state_fips'].unique())
fips_ok = actual_fips == expected_fips
print(f"[{'PASS' if fips_ok else 'FAIL'}] State FIPS values: {sorted(actual_fips)}")
if not fips_ok:
    print(f"  Expected: {sorted(expected_fips)}")
    print(f"  Extra: {actual_fips - expected_fips}")
    print(f"  Missing: {expected_fips - actual_fips}")

# 4. Duplicate check (block_geoid + provider_id + technology should be unique)
dup_count = combined.duplicated(subset=['block_geoid', 'provider_id', 'technology']).sum()
print(f"[{'PASS' if dup_count == 0 else 'WARN'}] Duplicate (block, provider, tech) combos: {dup_count:,}")

# 5. Null holding_company
null_company = combined['holding_company'].isna().sum()
print(f"[{'PASS' if null_company == 0 else 'FAIL'}] Null holding_company: {null_company}")

# 6. Null technology_name
null_tech = combined['technology_name'].isna().sum()
print(f"[{'PASS' if null_tech == 0 else 'FAIL'}] Null technology_name: {null_tech}")

# 7. Speed sanity
print(f"\n--- Speed Ranges ---")
print(f"Download: {combined['max_download'].min()} - {combined['max_download'].max()}")
print(f"Upload:   {combined['max_upload'].min()} - {combined['max_upload'].max()}")

=== VALIDATION REPORT ===

[PASS] Null block_geoid: 0


[PASS] Invalid block_geoid length: 0
[PASS] State FIPS values: ['06', '13', '17', '36', '48']


[PASS] Duplicate (block, provider, tech) combos: 0
[PASS] Null holding_company: 0
[PASS] Null technology_name: 0

--- Speed Ranges ---
Download: 0 - 1200000
Upload:   0 - 1200000


## 5. Export Combined CSV

In [8]:
output_path = '../output/all_states_block_level_availability.csv'
combined.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")
print(f"Rows: {combined.shape[0]:,}")
print(f"Columns: {combined.shape[1]}")

file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
print(f"File size: {file_size_mb:.1f} MB")

Saved to: ../output/all_states_block_level_availability.csv
Rows: 3,618,304
Columns: 13
File size: 365.6 MB


In [9]:
### Quick verify: re-read and confirm row count
verify = pd.read_csv(output_path, dtype=DTYPES, nrows=5)
print("Column order in exported CSV:")
print(list(verify.columns))
print("\nFirst 5 rows:")
verify

Column order in exported CSV:
['block_geoid', 'state_fips', 'county_fips', 'tract_fips', 'provider_id', 'holding_company', 'technology', 'technology_name', 'max_download', 'max_upload', 'serves_residential', 'serves_business', 'low_latency']

First 5 rows:


,block_geoid,state_fips,county_fips,tract_fips,provider_id,holding_company,technology,technology_name,max_download,max_upload,serves_residential,serves_business,low_latency
0,060014001001007,06,06001,06001400100,130317,Comcast Corporation,40,Cable,2000,250,True,True,1
1,060014001001009,06,06001,06001400100,130077,AT&T Inc.,10,Copper Wire (DSL),25,5,True,True,1
2,060014001001009,06,06001,06001400100,130317,Comcast Corporation,40,Cable,1200,35,True,True,1
3,060014001001010,06,06001,06001400100,130077,AT&T Inc.,10,Copper Wire (DSL),0,0,True,True,1
4,060014001001010,06,06001,06001400100,130077,AT&T Inc.,50,Fiber to the Premises (FTTP),5000,5000,True,True,1
